<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" width="400">

# Student Mark Sheet PDF Generation from Multiple Data Sources Practice Notebook

**Main Tutorial Link:** [Create Mark sheet pdf file by using data from SQLite / MySQL database table in Python ReportLab](https://www.plus2net.com/python/pdf-record.php)

---

## 1. Environment Setup & Datasets Download

Install `reportlab`, `sqlalchemy`, `openpyxl`, and `pandas` in Google Colab and download standard student sample files (`student.csv`, `student.xlsx`, `student.json`, `student.xml`, `student.sql`) from [plus2net pandas student tutorial](https://www.plus2net.com/python/pandas-student.php).

In [ ]:
!pip install reportlab sqlalchemy openpyxl pandas > /dev/null
!mkdir -p output data

# Download sample datasets
!wget -q -O data/student.csv https://www.plus2net.com/python/download/student.csv
!wget -q -O data/student.xlsx https://www.plus2net.com/python/download/student.xlsx
!wget -q -O data/student.json https://www.plus2net.com/python/download/student.json
!wget -q -O data/student.xml https://www.plus2net.com/python/download/student.xml
!wget -q -O data/student.sql https://www.plus2net.com/python/download/student.sql

print("Environment initialized and all 5 student datasets downloaded successfully!")

## 2. Define Mark Sheet Layout Template (`temp_marksheet`)

Define `my_temp(canvas)` to set up headers, institutional branding, date formatting, footer dividers, and diagonal background watermarks.

In [ ]:
from reportlab.lib.units import inch
from datetime import date

def my_temp(c):
    c.translate(inch, inch)
    c.setFont("Helvetica", 14)
    c.setStrokeColorRGB(0.1, 0.8, 0.1)
    c.setFillColorRGB(0, 0, 1)
    c.drawString(0, 9 * inch, "1234, ABCD Road")
    c.drawString(0, 8.7 * inch, "Mycity, ZIP : 12345")
    c.setFillColorRGB(0, 0.5, 1)
    c.drawString(2 * inch, 8.7 * inch, " www.plus2net.com ")
    c.setFillColorRGB(0, 0, 0)
    c.line(0, 8.6 * inch, 6.8 * inch, 8.6 * inch)
    
    dt = date.today().strftime('%d-%b-%Y')
    c.drawString(5.6 * inch, 9.3 * inch, dt)
    c.setFont("Helvetica", 8)
    c.drawString(3 * inch, 9.6 * inch, 'End Semester Examination')
    c.line(0, -0.7 * inch, 6.8 * inch, -0.7 * inch)
    c.setFillColorRGB(1, 0, 0)
    c.drawString(0, -0.9 * inch, "\u00A9 plus2net.com")
    
    # Watermark
    c.rotate(45)
    c.setFillColorCMYK(0, 0, 0, 0.08)
    c.setFont("Helvetica", 100)
    c.drawString(2 * inch, 1 * inch, "SAMPLE")
    c.rotate(-45)
    return c

print("Template function ready!")

## 3. Generate Mark Sheet from SQL Dump (`student.sql` via SQLite)

Populate an in-memory SQLite database from `student.sql`, fetch student ID `6` (`Alex John`), compute letter grade, and render canvas PDF.

In [ ]:
import sqlite3
from pathlib import Path
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
pdf_sql = output_dir / 'marksheet_from_sql.pdf'

# Populate SQLite database from SQL script
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
with open('data/student.sql', 'r', encoding='utf-8') as f:
    cursor.executescript(f.read())

cursor.execute("SELECT * FROM student WHERE id = 6")
data = cursor.fetchone() # (6, 'Alex John', 'Four', 55, 'male')

# Render Marksheet
c = canvas.Canvas(str(pdf_sql), pagesize=letter)
c = my_temp(c)

c.setFillColorRGB(1, 0, 0)
c.setFont("Helvetica", 70)
c.drawRightString(6.4 * inch, 7.5 * inch, 'MARKSHEET')

c.setFillColorRGB(0, 0, 0)
c.setFont("Helvetica", 24)
c.drawRightString(2.5 * inch, 6.8 * inch, 'ID:')
c.drawRightString(2.5 * inch, 6 * inch, 'Name:')
c.drawRightString(2.5 * inch, 5 * inch, 'Class:')
c.drawRightString(2.5 * inch, 4 * inch, 'Gender:')
c.drawRightString(2.5 * inch, 3 * inch, 'Mark:')
c.drawRightString(3 * inch, 2 * inch, 'Grade:')
c.drawRightString(6 * inch, -0.4 * inch, 'Signature')

c.setFillColorRGB(0, 0, 1)
c.setFont("Helvetica", 20)
c.drawString(3 * inch, 6.8 * inch, str(data[0]))
c.drawString(3 * inch, 6 * inch, str(data[1]))
c.drawString(3 * inch, 5 * inch, str(data[2]))
c.drawString(3 * inch, 4 * inch, str(data[4])) # Gender column
c.drawString(3 * inch, 3 * inch, str(data[3])) # Mark column

mark = int(data[3])
if mark >= 80:
    c.drawString(4 * inch, 2 * inch, 'A')
elif mark >= 60:
    c.drawString(4 * inch, 2 * inch, 'B')
else:
    c.drawString(4 * inch, 2 * inch, 'C')

c.showPage()
c.save()
print(f"SQL Dump Mark Sheet generated: {pdf_sql}")

## 4. Generate Mark Sheet from CSV Data (`student.csv`)

Read `student.csv` and output a mark sheet for student ID `1` (`John Deo`).

In [ ]:
import csv

pdf_csv = output_dir / 'marksheet_from_csv.pdf'
with open('data/student.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    students = list(reader)

st = next(s for s in students if s['id'] == '1')

c = canvas.Canvas(str(pdf_csv), pagesize=letter)
c = my_temp(c)

c.setFillColorRGB(1, 0, 0)
c.setFont("Helvetica", 70)
c.drawRightString(6.4 * inch, 7.5 * inch, 'MARKSHEET')

c.setFillColorRGB(0, 0, 0)
c.setFont("Helvetica", 24)
c.drawRightString(2.5 * inch, 6.8 * inch, 'ID:')
c.drawRightString(2.5 * inch, 6 * inch, 'Name:')
c.drawRightString(2.5 * inch, 5 * inch, 'Class:')
c.drawRightString(2.5 * inch, 4 * inch, 'Gender:')
c.drawRightString(2.5 * inch, 3 * inch, 'Mark:')
c.drawRightString(3 * inch, 2 * inch, 'Grade:')
c.drawRightString(6 * inch, -0.4 * inch, 'Signature')

c.setFillColorRGB(0, 0, 1)
c.setFont("Helvetica", 20)
c.drawString(3 * inch, 6.8 * inch, st['id'])
c.drawString(3 * inch, 6 * inch, st['name'])
c.drawString(3 * inch, 5 * inch, st['class'])
c.drawString(3 * inch, 4 * inch, st['gender'])
c.drawString(3 * inch, 3 * inch, st['mark'])

mark = int(st['mark'])
if mark >= 80:
    c.drawString(4 * inch, 2 * inch, 'A')
elif mark >= 60:
    c.drawString(4 * inch, 2 * inch, 'B')
else:
    c.drawString(4 * inch, 2 * inch, 'C')

c.showPage()
c.save()
print(f"CSV Mark Sheet generated: {pdf_csv}")

## 5. Generate Mark Sheets from Excel (`student.xlsx`) and JSON (`student.json`)

Demonstrate reading student records from Excel sheets and JSON arrays to produce customized mark sheets.

In [ ]:
import openpyxl
import json

# 1. Read Excel Record
wb = openpyxl.load_workbook('data/student.xlsx')
sheet = wb.active
rows = list(sheet.iter_rows(values_only=True))
records = rows[1:]
excel_st = next(r for r in records if r[0] == 2) # Max Ruin

pdf_excel = output_dir / 'marksheet_from_excel.pdf'
c = canvas.Canvas(str(pdf_excel), pagesize=letter)
c = my_temp(c)
c.setFillColorRGB(1, 0, 0)
c.setFont("Helvetica", 70)
c.drawRightString(6.4 * inch, 7.5 * inch, 'MARKSHEET')
c.setFillColorRGB(0, 0, 0)
c.setFont("Helvetica", 24)
c.drawRightString(2.5 * inch, 6.8 * inch, 'ID:')
c.drawRightString(2.5 * inch, 6 * inch, 'Name:')
c.drawRightString(2.5 * inch, 5 * inch, 'Class:')
c.drawRightString(2.5 * inch, 4 * inch, 'Gender:')
c.drawRightString(2.5 * inch, 3 * inch, 'Mark:')
c.drawRightString(3 * inch, 2 * inch, 'Grade:')

c.setFillColorRGB(0, 0, 1)
c.setFont("Helvetica", 20)
c.drawString(3 * inch, 6.8 * inch, str(excel_st[0]))
c.drawString(3 * inch, 6 * inch, str(excel_st[1]))
c.drawString(3 * inch, 5 * inch, str(excel_st[2]))
c.drawString(3 * inch, 4 * inch, str(excel_st[4]))
c.drawString(3 * inch, 3 * inch, str(excel_st[3]))
mark = int(excel_st[3])
c.drawString(4 * inch, 2 * inch, 'A' if mark >= 80 else ('B' if mark >= 60 else 'C'))
c.showPage()
c.save()
print(f"Excel Mark Sheet generated: {pdf_excel}")

---
### Visit [www.plus2net.com](https://www.plus2net.com) for more Python ReportLab tutorials and downloadable source code.